# Telecom KPI & Reference Data ETL Pipeline
Notebook ini digunakan untuk melakukan proses Automasi ETL (Extract, Transform, Load) dari file mentah jaringan (GSM, LTE, NR, Master Tracker, Sector) ke database MSSQL Server.

In [ ]:
import os
import glob
import re
import pandas as pd
import numpy as np
from sqlalchemy import create_engine, text, event
from urllib.parse import quote_plus
from dotenv import load_dotenv

# Memuat environment variables dari file .env
load_dotenv()
print("⚡ Libraries dan Environment berhasil dimuat.")

In [ ]:
# --- CONFIGURATION FROM ENVIRONMENT VARIABLES ---
ref_path = os.getenv("REF_PATH", "ref/") 
DB_NAME = os.getenv("DB_NAME", "XLSMOCN")
SCHEMA_NAME = "ref"

server = os.getenv("DB_SERVER")
database = os.getenv("DB_NAME")
username = os.getenv("DB_USER")
password = os.getenv("DB_PASSWORD")

# Validasi pengecekan konfigurasi env
if not all([server, database, username, password]):
    raise ValueError("❌ Error: Kredensial database tidak lengkap di file .env!")

# Membuat Connection String (ODBC Driver 17/18)
connection_string = (
    f"DRIVER={{ODBC Driver 17 for SQL Server}};"
    f"SERVER={server};"
    f"DATABASE={database};"
    f"UID={username};"
    f"PWD={password};"
)

odbc_str = quote_plus(connection_string)
engine = create_engine(f"mssql+pyodbc:///?odbc_connect={odbc_str}")

# AKTIFKAN FAST_EXECUTEMANY DENGAN PROTEKSI ATRIBUT
@event.listens_for(engine, "before_cursor_execute")
def receive_before_cursor_execute(context, connection, cursor, statement, parameters, context_executemany):
    if context_executemany:
        if hasattr(cursor, 'fast_executemany'):
            cursor.fast_executemany = True

print("🔄 Koneksi Database Remote + Fast Execution Berhasil Dikonfigurasi.")

## Helper Functions
Bagian ini berisi fungsi untuk membersihkan nama kolom, mengambil skema tabel dari database, serta melakukan penyesuaian tipe data (*data coercion*).

In [ ]:
def bersihkan_nama_kolom(nama_kolom):
    nama = str(nama_kolom)
    pola_tanggal = r"\(?\d{1,4}[-/][A-Za-z0-9]{2,3}[-/]\d{2,4}\)?|\(?\d{2,4}[-/]\d{2}[-/]\d{2}\)?|\(?[A-Za-z]{3}[-/]\d{2}\)?"
    nama = re.sub(pola_tanggal, "", nama).lower()
    nama = re.sub(r"[\s\-]+", "_", nama)
    nama = re.sub(r"[^a-z0-9_]", "", nama)
    nama = re.sub(r"_+", "_", nama).strip("_")
    return nama if nama else "kolom_tanpa_nama"

def get_sql_table_schema(table_name, engine, schema="ref"):
    query = f"""
        SELECT COLUMN_NAME, DATA_TYPE, CHARACTER_MAXIMUM_LENGTH
        FROM INFORMATION_SCHEMA.COLUMNS
        WHERE TABLE_SCHEMA = '{schema}' AND TABLE_NAME = '{table_name}'
    """
    with engine.connect() as conn:
        df_schema = pd.read_sql(query, conn)
    return df_schema

def coerce_date_columns(df, table_name, df_schema):
    date_cols = df_schema[df_schema['DATA_TYPE'].str.lower() == 'date']['COLUMN_NAME'].tolist()
    for col in date_cols:
        if col in df.columns:
            df[col] = df[col].astype(str).str.strip().replace(['nan', 'NULL', 'null', ''], np.nan)
            df[col] = pd.to_datetime(df[col], errors='coerce')
            df[col] = df[col].where(df[col].notnull(), None)
            df[col] = df[col].apply(lambda x: x.strftime('%Y-%m-%d') if isinstance(x, pd.Timestamp) else x)
    return df

def truncate_over_length_columns(df, table_name, df_schema):
    string_cols = df_schema[df_schema['DATA_TYPE'].str.contains('char', case=False, na=False)]
    for _, row in string_cols.iterrows():
        col_name = row['COLUMN_NAME']
        raw_max_len = row['CHARACTER_MAXIMUM_LENGTH']
        if pd.notna(raw_max_len) and raw_max_len > 0:
            max_len = int(raw_max_len)
            if col_name in df.columns:
                df[col_name] = df[col_name].apply(
                    lambda x: str(x)[:max_len] if (pd.notna(x) and x is not None) else x
                )
    return df

In [ ]:
def run_etl_pipeline(table_name, df_source, engine, ignore_mismatch=True, schema="ref"):
    if df_source is None or df_source.empty:
        print(f"[SKIP] Dataframe untuk tabel '{schema}.{table_name}' kosong.")
        return
        
    print(f"\n[RUNNING ETL] ==> Memulai proses untuk tabel: {schema}.{table_name}")
    df = df_source.copy()
    df.columns = [bersihkan_nama_kolom(c) for c in df.columns]
    
    unnamed_cols = [c for c in df.columns if "unnamed" in c]
    if unnamed_cols:
        df = df.drop(columns=unnamed_cols)
        
    df_schema = get_sql_table_schema(table_name, engine, schema)
    sql_cols = df_schema['COLUMN_NAME'].tolist()
    
    if not sql_cols:
        print(f"[ERROR] Tabel '{schema}.{table_name}' tidak terdeteksi di database.")
        return
        
    df_cols = df.columns.tolist()
    missing_in_sql = list(set(df_cols) - set(sql_cols))
    missing_in_df = list(set(sql_cols) - set(df_cols))
    
    if missing_in_sql or missing_in_df:
        print(f"⚠️ [WARNING] Ketidakcocokan skema pada tabel '{table_name}':")
        if missing_in_sql: print(f"   - Di Dataframe saja: {missing_in_sql}")
        if missing_in_df: print(f"   - Di SQL saja: {missing_in_df}")
        
        if ignore_mismatch:
            print("   -> Menyesuaikan kolom dengan struktur SQL...")
            df = df[[c for c in df_cols if c in sql_cols]]
        else:
            print("   -> Proses dihentikan demi integritas data.")
            return

    df = coerce_date_columns(df, table_name, df_schema)
    df = truncate_over_length_columns(df, table_name, df_schema)
    
    try:
        with engine.begin() as conn:
            conn.execute(text(f"TRUNCATE TABLE {schema}.{table_name};"))
            print(f"   -> Sukses TRUNCATE TABLE {schema}.{table_name}.")
            
        df.to_sql(name=table_name, con=engine, schema=schema, if_exists='append', index=False, chunksize=10000)
        print(f"   -> Selesai! Berhasil mengupload {len(df):,} baris data.")
    except Exception as e:
        print(f"❌ [CRITICAL ERROR] Gagal upload ke {table_name}: {str(e)}")

## 1. Extraction Phase
Membaca file mentah dari direktori lokal dan memuatnya ke dalam Pandas Dataframe.

In [ ]:
all_files = glob.glob(os.path.join(ref_path, "*"))
GSM_Cell, LTE_Cell, NR_Cell, Master_MOCN, Master_NEWSITE, Sector = None, None, None, None, None, None

# 1. Ekstrak GSM
gsm_match = [f for f in all_files if os.path.basename(f).startswith("GSM_Cell_Report")]
if gsm_match:
    print(f"[EXTRACT] Membaca GSM File: {os.path.basename(gsm_match[0])}")
    GSM_Cell = pd.read_csv(gsm_match[0], dtype=str)

# 2. Ekstrak LTE
lte_match = [f for f in all_files if os.path.basename(f).startswith("LTE_Cell_Report")]
if lte_match:
    print(f"[EXTRACT] Membaca LTE File: {os.path.basename(lte_match[0])}")
    LTE_Cell = pd.read_csv(lte_match[0], dtype=str)

# 3. Ekstrak NR
nr_match = [f for f in all_files if os.path.basename(f).startswith("NR_Cell_Report")]
if nr_match:
    print(f"[EXTRACT] Membaca NR File: {os.path.basename(nr_match[0])}")
    NR_Cell = pd.read_csv(nr_match[0], dtype=str)

# 4. Ekstrak Master Tracker
master_match = [f for f in all_files if os.path.basename(f).startswith("00. Master Tracker MOCN")]
if master_match:
    print(f"[EXTRACT] Membaca Master File: {os.path.basename(master_match[0])}")
    Master_MOCN = pd.read_excel(master_match[0], sheet_name="Master Tracker", skiprows=1, dtype=str)
    Master_NEWSITE = pd.read_excel(master_match[0], sheet_name="New Site", skiprows=1, dtype=str)

In [ ]:
# 5. Ekstrak & Gabung Sector (4G & 5G)
f_non_jabo = [f for f in all_files if "Non JABO" in os.path.basename(f)]
f_new_site = [f for f in all_files if "New Site" in os.path.basename(f)]
f_hq_msite = [f for f in all_files if "(HQ) Msite" in os.path.basename(f)]

if f_non_jabo and f_new_site and f_hq_msite:
    print("[EXTRACT] Menggabungkan data Sector 4G & 5G dari 3 File XLSB...")
    
    # Komponen Sector 4G
    d41 = pd.read_excel(f_non_jabo[0], sheet_name="EP 4G", engine="pyxlsb", usecols=["Cell Name After", "Sector ID"], dtype=str).rename(columns={"Cell Name After": "moentity", "Sector ID": "sector"})
    d41["priority"] = 1
    d42 = pd.read_excel(f_new_site[0], sheet_name="4G", engine="pyxlsb", usecols=["Cell Name After", "Sector ID"], dtype=str).rename(columns={"Cell Name After": "moentity", "Sector ID": "sector"})
    d42["priority"] = 2
    d43 = pd.read_excel(f_hq_msite[0], sheet_name="4G", engine="pyxlsb", usecols=["Cellname", "Sector"], dtype=str).rename(columns={"Cellname": "moentity", "Sector": "sector"})
    d43["priority"] = 3
    s4 = pd.concat([d41, d42, d43], ignore_index=True).sort_values("priority").drop_duplicates(subset=["moentity"], keep="first")
    s4["teknologi"] = "4G"
    
    # Komponen Sector 5G
    d51 = pd.read_excel(f_non_jabo[0], sheet_name="EP 5G", engine="pyxlsb", skiprows=1, usecols=["Cell Name After", "Sector ID"], dtype=str).rename(columns={"Cell Name After": "moentity", "Sector ID": "sector"})
    d51["priority"] = 1
    d52 = pd.read_excel(f_new_site[0], sheet_name="5G", engine="pyxlsb", usecols=["Cell Name After", "Sector ID"], dtype=str).rename(columns={"Cell Name After": "moentity", "Sector ID": "sector"})
    d52["priority"] = 2
    d53 = pd.read_excel(f_hq_msite[0], sheet_name="5G", engine="pyxlsb", usecols=["CELLNAME", "SECTOR ID"], dtype=str).rename(columns={"CELLNAME": "moentity", "SECTOR ID": "sector"})
    d53["priority"] = 3
    s5 = pd.concat([d51, d52, d53], ignore_index=True).sort_values("priority").drop_duplicates(subset=["moentity"], keep="first")
    s5["teknologi"] = "5G"
    
    Sector = pd.concat([s4, s5], ignore_index=True).drop(columns=["priority"])
    print("✓ Penggabungan Sector selesai.")

In [ ]:
# 5. Ekstrak & Gabung Sector (4G & 5G)
f_non_jabo = [f for f in all_files if "Non JABO" in os.path.basename(f)]
f_new_site = [f for f in all_files if "New Site" in os.path.basename(f)]
f_hq_msite = [f for f in all_files if "(HQ) Msite" in os.path.basename(f)]

if f_non_jabo and f_new_site and f_hq_msite:
    print("[EXTRACT] Menggabungkan data Sector 4G & 5G dari 3 File XLSB...")
    
    # Komponen Sector 4G
    d41 = pd.read_excel(f_non_jabo[0], sheet_name="EP 4G", engine="pyxlsb", usecols=["Cell Name After", "Sector ID"], dtype=str).rename(columns={"Cell Name After": "moentity", "Sector ID": "sector"})
    d41["priority"] = 1
    d42 = pd.read_excel(f_new_site[0], sheet_name="4G", engine="pyxlsb", usecols=["Cell Name After", "Sector ID"], dtype=str).rename(columns={"Cell Name After": "moentity", "Sector ID": "sector"})
    d42["priority"] = 2
    d43 = pd.read_excel(f_hq_msite[0], sheet_name="4G", engine="pyxlsb", usecols=["Cellname", "Sector"], dtype=str).rename(columns={"Cellname": "moentity", "Sector": "sector"})
    d43["priority"] = 3
    s4 = pd.concat([d41, d42, d43], ignore_index=True).sort_values("priority").drop_duplicates(subset=["moentity"], keep="first")
    s4["teknologi"] = "4G"
    
    # Komponen Sector 5G
    d51 = pd.read_excel(f_non_jabo[0], sheet_name="EP 5G", engine="pyxlsb", skiprows=1, usecols=["Cell Name After", "Sector ID"], dtype=str).rename(columns={"Cell Name After": "moentity", "Sector ID": "sector"})
    d51["priority"] = 1
    d52 = pd.read_excel(f_new_site[0], sheet_name="5G", engine="pyxlsb", usecols=["Cell Name After", "Sector ID"], dtype=str).rename(columns={"Cell Name After": "moentity", "Sector ID": "sector"})
    d52["priority"] = 2
    d53 = pd.read_excel(f_hq_msite[0], sheet_name="5G", engine="pyxlsb", usecols=["CELLNAME", "SECTOR ID"], dtype=str).rename(columns={"CELLNAME": "moentity", "SECTOR ID": "sector"})
    d53["priority"] = 3
    s5 = pd.concat([d51, d52, d53], ignore_index=True).sort_values("priority").drop_duplicates(subset=["moentity"], keep="first")
    s5["teknologi"] = "5G"
    
    Sector = pd.concat([s4, s5], ignore_index=True).drop(columns=["priority"])
    print("✓ Penggabungan Sector selesai.")

## 2. Load Phase (Database Ingestion)
Menjalankan pipeline ETL secara sekuensial untuk mengirim semua dataframe ke tabel MSSQL yang sesuai.

In [ ]:
print("--- MEMULAI PUSH DATA PIPELINE KE MSSQL SERVER ---")

run_etl_pipeline("gsm_cell", GSM_Cell, engine, ignore_mismatch=True)
run_etl_pipeline("lte_cell", LTE_Cell, engine, ignore_mismatch=True)
run_etl_pipeline("nr_cell", NR_Cell, engine, ignore_mismatch=True)
run_etl_pipeline("master_mocn", Master_MOCN, engine, ignore_mismatch=True)
run_etl_pipeline("master_newsite", Master_NEWSITE, engine, ignore_mismatch=True)
run_etl_pipeline("sector", Sector, engine, ignore_mismatch=True)

print("\n🎉 ALL PIPELINE EXECUTED SUCCESSFULLY!")